###### Step 1: Imports and configuration

In [0]:
#Install Vector Search client
%pip install databricks-vectorsearch
dbutils.library.restartPython()

###### Check existing AI Search endpoints

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole


w = WorkspaceClient()
vsc = VectorSearchClient()
vsc.list_endpoints()

###### Create endpoint only if missing

In [0]:
VECTOR_SEARCH_ENDPOINT_NAME = "telco-vector-search-endpoint"

try:
    endpoint = vsc.get_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print("Endpoint already exists")

except Exception:
    print("Creating endpoint...")

    vsc.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )

###### Wait until endpoint is online

In [0]:
endpoint = vsc.get_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
print(endpoint)

###### Check indexes

In [0]:
vsc.list_indexes(VECTOR_SEARCH_ENDPOINT_NAME)

###### Create index only if needed

In [0]:
SOURCE_TABLE = "dbw_agentic_ai_dev.telco_ai.customer_note_embeddings"
INDEX_NAME =  "dbw_agentic_ai_dev.telco_ai.customer_note_embeddings_index"

embedding_dimension = len(
    spark.table(SOURCE_TABLE)
         .select("embedding")
         .first()["embedding"]
)

vsc.create_delta_sync_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    source_table_name=SOURCE_TABLE,
    index_name=INDEX_NAME,
    pipeline_type="TRIGGERED",
    primary_key="customer_id",
    embedding_dimension=embedding_dimension,
    embedding_vector_column="embedding",
    columns_to_sync=[
        "customer_id",
        "note"
    ]
)

###### Monitor status

In [0]:
index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=INDEX_NAME
)

print(index.describe())

###### Quick Search Test

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

question = "customer wants to cancel"

response = w.serving_endpoints.query(
    name="databricks-gte-large-en",
    input=[question]
)

question_embedding = [float(x) for x in response.data[0].embedding]

results = index.similarity_search(
    query_vector=question_embedding,
    columns=["customer_id", "note"],
    num_results=3
)

print(results)

###### Step 2: Get the index

In [0]:
index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=INDEX_NAME
)
EMBEDDING_MODEL = "databricks-gte-large-en"
LLM_MODEL = "databricks-meta-llama-3-1-8b-instruct"

###### Step 3: Create a retrieval function

In [0]:
def retrieve_context(question, num_results=3):
    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL,
        input=[question]
    )

    question_embedding = [
        float(x) for x in response.data[0].embedding
    ]

    results = index.similarity_search(
        query_vector=question_embedding,
        columns=["customer_id", "note"],
        num_results=num_results
    )

    rows = results["result"]["data_array"]

    context_lines = []
    for customer_id, note, score in rows:
        context_lines.append(
            f"Customer {int(customer_id)}: {note}"
        )

    return "\n".join(context_lines), rows

###### Step 4: Build the prompt

In [0]:
def build_prompt(question, context):
    prompt = f"""
You are a helpful telco customer support analyst.

Answer the user's question using only the context below.
If the context does not contain enough information, say:
"I do not have enough information from the provided context."

Context:
{context}

Question:
{question}

Answer:
"""
    return prompt

###### Step 5: Call the LLM

In [0]:
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

def generate_answer(prompt):
    response = w.serving_endpoints.query(
        name=LLM_MODEL,
        messages=[
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt
            )
        ],
        max_tokens=300,
        temperature=0.0
    )

    return response.choices[0].message.content

###### Step 6: End-to-end RAG function

In [0]:
def rag_answer(question):
    context, retrieved_rows = retrieve_context(
        question=question,
        num_results=3
    )

    prompt = build_prompt(
        question=question,
        context=context
    )

    answer = generate_answer(prompt)

    return answer, context, retrieved_rows

###### Step 7: Test

In [0]:
question = "Why are customers likely to cancel service?"

answer, context, retrieved_rows = rag_answer(question)

print("QUESTION:")
print(question)

print("\nRETRIEVED CONTEXT:")
print(context)

print("\nANSWER:")
print(answer)

In [0]:
#GPT OSS 20B returned reasoning-style output.
#Llama 3.1 8B Instruct returned cleaner final-answer output.